# Playground — fixed family and model-size comparison

This is a validation-first benchmark of pre-registered HOG, scratch-CNN, and pretrained families/sizes on the same raw split as the primary notebook. It reports the held-out test set only for the validation-selected row. Use `COMPCLASS_PLAYGROUND_MODELS` to run a smaller explicit list, and use `COMPCLASS_PROFILE=smoke` for a fast pipeline check.

Every completed model row saves a metadata-bearing checkpoint in `outputs/weights/`; the session contract and validation table are saved under `outputs/metrics/`.


## 0. Setup

Recommended: from the repository root run `cd notebooks && COMPCLASS_PROFILE=paper uv run jupyter lab`. Set `COMPCLASS_WORK_DIR` only when the kernel opens with a different working directory. Run the next cell in the selected kernel; it includes a uv path and a pip fallback.


In [ ]:
import importlib.util
import shutil
import subprocess
import sys

REQUIREMENTS = [
    "numpy>=1.26,<3",
    "pandas>=2.2,<3",
    "matplotlib>=3.9,<4",
    "seaborn>=0.13,<1",
    "pillow>=10,<12",
    "scikit-image>=0.24,<1",
    "scikit-learn>=1.5,<2",
    "torch>=2.4,<3",
    "torchvision>=0.19,<1",
    "tqdm>=4.66,<5",
    "joblib>=1.4,<2",
    "kaggle>=1.6,<2",
    "rich>=13.9,<15",
    "ipykernel>=6.29,<7",
    "jupyterlab>=4,<5",
    "notebook>=7.2,<8",
]
IMPORTS = {
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "seaborn": "seaborn", "pillow": "PIL", "scikit-image": "skimage",
    "scikit-learn": "sklearn", "torch": "torch", "torchvision": "torchvision",
    "joblib": "joblib", "kaggle": "kaggle", "rich": "rich",
}
missing = [package for package, module in IMPORTS.items() if importlib.util.find_spec(module) is None]
if missing:
    command = (
        ["uv", "pip", "install", "--python", sys.executable, *REQUIREMENTS]
        if shutil.which("uv")
        else [sys.executable, "-m", "pip", "install", *REQUIREMENTS]
    )
    print("Missing:", ", ".join(missing))
    print("Running:", " ".join(command))
    subprocess.check_call(command)
else:
    print("Core dependencies are already available in this kernel.")


## 1. Helpers and configuration


In [ ]:
# ===== Self-contained helpers (no external project package) =====
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore", message="CUDA initialization: Unexpected error")

import json
import os
import random
import time
import zipfile
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from skimage.color import rgb2gray
from skimage.feature import hog
from skimage.transform import resize
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

# Paths: project root (parent of notebooks/) so data/ is shared
ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
SPLITS_DIR = DATA_DIR / "splits"
OUTPUTS_DIR = ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
WEIGHTS_DIR = OUTPUTS_DIR / "weights"
METRICS_DIR = OUTPUTS_DIR / "metrics"

KAGGLE_DATASET = "mdfaisalahmed025/electronic-component-recognizer"
DATASET_DIRNAME = "electronic-component-recognizer"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
SEED = 42
IMAGE_SIZE = 224
SPLIT_RATIOS = (0.70, 0.15, 0.15)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def ensure_output_dirs() -> None:
    for path in (RAW_DIR, SPLITS_DIR, FIGURES_DIR, WEIGHTS_DIR, METRICS_DIR):
        path.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class Sample:
    path: str
    label: str


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _is_image(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def _dir_has_images(path: Path) -> bool:
    """True if this folder (or any nested folder) contains image files."""
    try:
        for p in path.rglob("*"):
            if _is_image(p):
                return True
    except PermissionError:
        return False
    return False


def find_imagefolder_root(search_root: Path) -> Path | None:
    """Find a directory whose children are class folders containing images (possibly nested)."""
    if not search_root.exists():
        return None
    candidates: list[Path] = []
    for directory in [search_root, *search_root.rglob("*")]:
        if not directory.is_dir():
            continue
        subdirs = [p for p in directory.iterdir() if p.is_dir() and not p.name.startswith(".")]
        if len(subdirs) < 2:
            continue
        class_like = sum(1 for sub in subdirs if _dir_has_images(sub))
        # Allow a couple of non-image sidecar folders, but require mostly class folders
        if class_like >= 2 and class_like >= len(subdirs) - 1:
            candidates.append(directory)
    if not candidates:
        return None
    candidates.sort(key=lambda p: (len(p.parts), -sum(1 for _ in p.iterdir() if _.is_dir())))
    return candidates[0]


def _extract_zips(raw_dir: Path) -> None:
    for zip_path in sorted(raw_dir.glob("*.zip")):
        extract_to = raw_dir / zip_path.stem
        if extract_to.exists() and find_imagefolder_root(extract_to):
            continue
        extract_to.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_to)


def _kaggle_auth_hint() -> str:
    return (
        "Kaggle authentication failed or is missing.\n"
        "Options:\n"
        "  1) `kaggle auth login` (OAuth)\n"
        "  2) Export KAGGLE_API_TOKEN from https://www.kaggle.com/settings/api\n"
        "  3) Save token to ~/.kaggle/access_token\n"
        "  4) Legacy: ~/.kaggle/kaggle.json\n"
        f"Or manual: drop the dataset zip/folder under {RAW_DIR}"
    )


def _download_kaggle(raw_dir: Path) -> Path:
    raw_dir.mkdir(parents=True, exist_ok=True)
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ImportError as exc:
        raise RuntimeError("Install kaggle or use the manual data drop.") from exc
    api = KaggleApi()
    try:
        api.authenticate()
    except Exception as exc:
        raise RuntimeError(_kaggle_auth_hint()) from exc
    print(f"Downloading {KAGGLE_DATASET} …")
    try:
        api.dataset_download_files(KAGGLE_DATASET, path=str(raw_dir), quiet=False)
    except Exception as exc:
        raise RuntimeError(f"Kaggle download failed.\n{_kaggle_auth_hint()}") from exc
    _extract_zips(raw_dir)
    root = find_imagefolder_root(raw_dir)
    if root is None:
        raise RuntimeError(f"Downloaded but no ImageFolder layout under {raw_dir}")
    return root


def resolve_dataset_root(*, prefer_kaggle: bool = True) -> Path:
    """Kaggle API if empty; else use zip/folder already in data/raw/."""
    ensure_output_dirs()
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    existing = find_imagefolder_root(RAW_DIR)
    if existing is not None:
        print(f"Using existing dataset at {existing}")
        return existing
    zips = list(RAW_DIR.glob("*.zip"))
    if zips:
        print(f"Found manual zip(s): {[z.name for z in zips]} — extracting …")
        _extract_zips(RAW_DIR)
        existing = find_imagefolder_root(RAW_DIR)
        if existing is not None:
            print(f"Using extracted dataset at {existing}")
            return existing
        raise RuntimeError(f"Zip extracted but no class folders found under {RAW_DIR}")
    if prefer_kaggle:
        return _download_kaggle(RAW_DIR)
    raise RuntimeError(
        f"No dataset found. Place a zip or ImageFolder under {RAW_DIR}, "
        "or enable Kaggle API credentials."
    )


def list_samples(dataset_root: Path, *, validate: bool = True) -> list[Sample]:
    """Collect images under each class folder (supports nested class/class/*.jpg layouts)."""
    samples: list[Sample] = []
    skipped = 0
    for class_dir in sorted(p for p in dataset_root.iterdir() if p.is_dir() and not p.name.startswith(".")):
        for image_path in sorted(class_dir.rglob("*")):
            if not _is_image(image_path):
                continue
            if any(part.startswith(".") for part in image_path.relative_to(class_dir).parts):
                continue
            if validate:
                try:
                    _open_rgb_image(image_path)
                except Exception:
                    skipped += 1
                    continue
            samples.append(Sample(path=str(image_path.resolve()), label=class_dir.name))
    if skipped:
        print(f"Skipped {skipped} unreadable/corrupt image(s)")
    if not samples:
        raise RuntimeError(f"No images found under {dataset_root}")
    return samples


def class_names_from_samples(samples: list[Sample]) -> list[str]:
    return sorted({s.label for s in samples})


def build_or_load_splits(
    samples: list[Sample],
    *,
    seed: int = SEED,
    ratios: tuple[float, float, float] = SPLIT_RATIOS,
    force: bool = False,
) -> dict[str, list[Sample]]:
    ensure_output_dirs()
    manifest_path = SPLITS_DIR / "split_manifest.json"
    meta_path = SPLITS_DIR / "split_meta.json"
    if manifest_path.exists() and not force:
        payload = json.loads(manifest_path.read_text())
        splits = {}
        dropped = 0
        for k, rows in payload.items():
            kept = []
            for row in rows:
                sample = Sample(**row)
                if not Path(sample.path).is_file():
                    dropped += 1
                    continue
                kept.append(sample)
            splits[k] = kept
        if dropped:
            print(f"Dropped {dropped} missing path(s) from frozen split")
        print(f"Loaded frozen split from {manifest_path}")
        return splits

    train_ratio, val_ratio, test_ratio = ratios
    paths = [s.path for s in samples]
    labels = [s.label for s in samples]
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        paths, labels, test_size=(1.0 - train_ratio), random_state=seed, stratify=labels
    )
    relative_val = val_ratio / (val_ratio + test_ratio)
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=(1.0 - relative_val), random_state=seed, stratify=temp_labels
    )
    splits = {
        "train": [Sample(path=p, label=y) for p, y in zip(train_paths, train_labels, strict=True)],
        "val": [Sample(path=p, label=y) for p, y in zip(val_paths, val_labels, strict=True)],
        "test": [Sample(path=p, label=y) for p, y in zip(test_paths, test_labels, strict=True)],
    }
    manifest_path.write_text(
        json.dumps({k: [{"path": s.path, "label": s.label} for s in v] for k, v in splits.items()}, indent=2)
    )
    counts = {}
    for split_name, split_samples in splits.items():
        bag: dict[str, int] = defaultdict(int)
        for sample in split_samples:
            bag[sample.label] += 1
        counts[split_name] = dict(sorted(bag.items()))
    meta_path.write_text(
        json.dumps(
            {
                "seed": seed,
                "ratios": ratios,
                "n_train": len(splits["train"]),
                "n_val": len(splits["val"]),
                "n_test": len(splits["test"]),
                "class_counts": counts,
            },
            indent=2,
        )
    )
    print(f"Wrote frozen split to {manifest_path}")
    return splits


def summarize_splits(splits: dict[str, list[Sample]]) -> None:
    for name, items in splits.items():
        print(f"  {name:5s}: {len(items):4d} images")


def _open_rgb_image(path: str | Path) -> Image.Image:
    """Open an image as RGB; raise if unreadable/corrupt."""
    with Image.open(path) as image:
        # Palette+transparency needs RGBA first, then RGB
        if image.mode in ("P", "RGBA", "LA"):
            image = image.convert("RGBA")
        image = image.convert("RGB")
        image.load()
        return image.copy()


def load_rgb(path: str | Path, size: int = IMAGE_SIZE) -> np.ndarray:
    image = _open_rgb_image(path)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return resize(array, (size, size), anti_aliasing=True, preserve_range=True).astype(np.float32)


def hog_vector(image_rgb: np.ndarray) -> np.ndarray:
    gray = rgb2gray(image_rgb)
    return hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
    ).astype(np.float32)


def extract_hog_matrix(
    samples: list[Sample],
    *,
    size: int = IMAGE_SIZE,
    show_progress: bool = True,
    stage: str = "HOG",
):
    labels = sorted({s.label for s in samples})
    label_to_idx = {name: i for i, name in enumerate(labels)}
    features, targets = [], []
    skipped = 0
    t0 = time.perf_counter()
    print(f"[{stage}] extracting features for {len(samples)} images …", flush=True)
    iterator = tqdm(samples, desc=stage) if show_progress else samples
    for sample in iterator:
        try:
            features.append(hog_vector(load_rgb(sample.path, size=size)))
            targets.append(label_to_idx[sample.label])
        except Exception:
            skipped += 1
            continue
    if skipped:
        print(f"[{stage}] skipped {skipped} unreadable image(s)", flush=True)
    if not features:
        raise RuntimeError(f"[{stage}] no readable images for HOG extraction")
    x = np.stack(features)
    y = np.asarray(targets, dtype=np.int64)
    print(
        f"[{stage}] done in {time.perf_counter() - t0:.1f}s  "
        f"shape={x.shape}  (~{x.nbytes / 1e6:.1f} MB)",
        flush=True,
    )
    return x, y, labels


def train_hog_svm(train_samples, val_samples=None, *, C: float = 1.0, max_iter: int = 2000) -> dict:
    """HOG + Linear SVM with explicit phase logs (so long fits don't look hung)."""
    overall_t0 = time.perf_counter()
    print("=" * 60, flush=True)
    print("CLASSICAL: HOG → LinearSVM", flush=True)
    print("=" * 60, flush=True)

    x_train, y_train, class_names = extract_hog_matrix(
        train_samples, stage="HOG train"
    )
    n_samples, n_features = x_train.shape
    # n_samples << n_features → dual=False is much faster than dual="auto"/True
    dual = bool(n_samples >= n_features)
    print(
        f"[SVM] fitting LinearSVC  samples={n_samples}  features={n_features}  "
        f"dual={dual}  max_iter={max_iter}  (verbose ticks = still busy)",
        flush=True,
    )

    pipeline = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "svm",
                LinearSVC(
                    C=C,
                    max_iter=max_iter,
                    dual=dual,
                    random_state=42,
                    verbose=1,  # prints liblinear iterations so silence ≠ hang
                ),
            ),
        ]
    )
    t0 = time.perf_counter()
    pipeline.fit(x_train, y_train)
    fit_seconds = time.perf_counter() - t0
    print(f"[SVM] fit done in {fit_seconds:.1f}s", flush=True)

    result = {
        "model": pipeline,
        "class_names": class_names,
        "train_seconds": time.perf_counter() - overall_t0,
        "fit_seconds": fit_seconds,
        "n_features": int(n_features),
    }
    if val_samples is not None:
        x_val, y_val, _ = extract_hog_matrix(val_samples, stage="HOG val")
        print("[SVM] scoring validation …", flush=True)
        y_hat = pipeline.predict(x_val)
        result["best_val_macro_f1"] = float(
            f1_score(
                y_val,
                y_hat,
                average="macro",
                labels=list(range(len(class_names))),
                zero_division=0,
            )
        )
        print(f"[SVM] val macro-F1={result['best_val_macro_f1']:.4f}", flush=True)

    coef = pipeline.named_steps["svm"].coef_
    intercept = pipeline.named_steps["svm"].intercept_
    result["params"] = int(coef.size + intercept.size)
    print(
        f"[CLASSICAL] total {result['train_seconds']:.1f}s  params≈{result['params']}",
        flush=True,
    )
    return result


def predict_hog_svm(pipeline, samples):
    x, y_true, class_names = extract_hog_matrix(samples, stage="HOG test")
    print("[SVM] predicting test set …", flush=True)
    y_pred = pipeline.predict(x).tolist()
    print("[SVM] predict done", flush=True)
    return y_true.tolist(), y_pred, class_names


class SmallCNN(nn.Module):
    def __init__(self, num_classes: int) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(256, 384, 3, padding=1), nn.BatchNorm2d(384), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(384, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.pool(self.features(x)))


def build_resnet18(num_classes: int, *, pretrained: bool = True) -> nn.Module:
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_transforms(train: bool, *, size: int = IMAGE_SIZE) -> transforms.Compose:
    if train:
        return transforms.Compose(
            [
                transforms.Resize((size, size)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(0.15, 0.15, 0.1),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ]
        )
    return transforms.Compose(
        [
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )


class ComponentDataset(Dataset):
    def __init__(self, samples, class_to_idx, transform=None):
        self.samples = samples
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = _open_rgb_image(sample.path)
        if self.transform is not None:
            image = self.transform(image)
        return image, self.class_to_idx[sample.label]


def make_loaders(splits, class_names, *, batch_size=16, num_workers=0):
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    loaders = {}
    for split_name, items in splits.items():
        ds = ComponentDataset(items, class_to_idx, transform=build_transforms(train=(split_name == "train")))
        loaders[split_name] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=(split_name == "train"),
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
        )
    return loaders


@torch.no_grad()
def predict_loader(model, loader, device, *, horizontal_flip=True):
    model.eval()
    y_true, y_pred = [], []
    for inputs, targets in loader:
        inputs = inputs.to(device)
        logits = model(inputs)
        if horizontal_flip:
            logits = (logits + model(torch.flip(inputs, dims=[3]))) / 2
        preds = logits.argmax(dim=1).cpu().tolist()
        y_pred.extend(preds)
        y_true.extend(targets.tolist())
    return y_true, y_pred


def macro_f1_from_logits(model, loader, device, num_classes: int) -> float:
    y_true, y_pred = predict_loader(model, loader, device)
    return float(f1_score(y_true, y_pred, average="macro", labels=list(range(num_classes)), zero_division=0))


def train_torch_model(
    model,
    loaders,
    *,
    device,
    num_classes,
    epochs=25,
    lr=1e-3,
    weight_decay=1e-4,
    patience=5,
    freeze_backbone_epochs=0,
):
    model = model.to(device)
    train_targets = [loaders["train"].dataset.class_to_idx[sample.label] for sample in loaders["train"].dataset.samples]
    counts = torch.bincount(torch.tensor(train_targets), minlength=num_classes).float()
    class_weights = (counts.mean() / counts).sqrt().to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.08)

    def set_backbone_trainable(trainable: bool) -> None:
        for name, param in model.named_parameters():
            if name.startswith("fc") or name.startswith("classifier"):
                param.requires_grad = True
            else:
                param.requires_grad = trainable

    if freeze_backbone_epochs > 0:
        set_backbone_trainable(False)

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    history = []
    best_f1 = -1.0
    best_state = None
    epochs_without_improve = 0
    t0 = time.perf_counter()

    for epoch in range(1, epochs + 1):
        if freeze_backbone_epochs > 0 and epoch == freeze_backbone_epochs + 1:
            set_backbone_trainable(True)
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr * 0.1, weight_decay=weight_decay)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

        model.train()
        running_loss, n_batches = 0.0, 0
        for inputs, targets in tqdm(loaders["train"], desc=f"epoch {epoch}/{epochs}", leave=False):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item())
            n_batches += 1

        train_loss = running_loss / max(n_batches, 1)
        val_f1 = macro_f1_from_logits(model, loaders["val"], device, num_classes)
        scheduler.step(val_f1)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_macro_f1": val_f1})
        print(f"epoch {epoch:02d}  loss={train_loss:.4f}  val_macro_f1={val_f1:.4f}")

        if val_f1 > best_f1 + 1e-4:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= patience:
                print(f"Early stopping at epoch {epoch} (best val macro-F1={best_f1:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return {
        "model": model,
        "history": history,
        "best_val_macro_f1": best_f1,
        "train_seconds": time.perf_counter() - t0,
        "seed": SEED,
    }


def compute_classification_metrics(y_true, y_pred, class_names) -> dict[str, Any]:
    labels = list(range(len(class_names)))
    per_class_f1 = f1_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0)),
        "per_class_f1": {name: float(s) for name, s in zip(class_names, per_class_f1, strict=True)},
        "classification_report": classification_report(
            y_true, y_pred, labels=labels, target_names=class_names, output_dict=True, zero_division=0
        ),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def plot_confusion_matrix(y_true, y_pred, class_names, *, title, save_path=None, figsize=(12, 10)):
    ensure_output_dirs()
    matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.xticks(rotation=45, ha="right")
    fig.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig


def metrics_table(rows):
    frame = pd.DataFrame(rows)
    preferred = ["model", "family", "accuracy", "macro_f1", "weighted_f1", "params", "train_seconds"]
    cols = [c for c in preferred if c in frame.columns] + [c for c in frame.columns if c not in preferred]
    return frame[cols].sort_values("macro_f1", ascending=False).reset_index(drop=True)


ensure_output_dirs()
set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0
BATCH_SIZE = 16 if DEVICE.type == "cuda" else 8
print(f"cwd={ROOT}")
print(f"device={DEVICE}  batch_size={BATCH_SIZE}  num_workers={NUM_WORKERS}")
print(f"data/raw → {RAW_DIR}")


In [ ]:
import hashlib

from collections import Counter

from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()
RUN_PROFILE = os.getenv("COMPCLASS_PROFILE", "standard")
PROFILES = {
    "smoke": {"max_per_class": 2, "epochs": 1, "patience": 1, "batch_size": 2, "image_size": 64, "trials": 1},
    "standard": {"max_per_class": None, "epochs": 10, "patience": 3, "batch_size": 16, "image_size": 224, "trials": 5},
    "paper": {"max_per_class": None, "epochs": 25, "patience": 5, "batch_size": 16, "image_size": 224, "trials": 20},
}
if RUN_PROFILE not in PROFILES:
    raise ValueError(f"Unknown profile {RUN_PROFILE!r}; choose {sorted(PROFILES)}")
PROFILE = PROFILES[RUN_PROFILE]
SEED = int(os.getenv("COMPCLASS_SEED", "42"))
SPLIT_RATIOS = (0.70, 0.15, 0.15)
IMAGE_SIZE = int(os.getenv("COMPCLASS_IMAGE_SIZE", PROFILE["image_size"]))
BATCH_SIZE = int(os.getenv("COMPCLASS_BATCH_SIZE", PROFILE["batch_size"]))
FORCE_RETRAIN = os.getenv("COMPCLASS_FORCE_RETRAIN", "").lower() in {"1", "true", "yes", "on"}
USE_PRETRAINED = os.getenv("COMPCLASS_USE_PRETRAINED", "1").lower() not in {"0", "false", "no"}
ALLOW_PRETRAINED_FALLBACK = RUN_PROFILE == "smoke"

TRAINING_VARIANT = os.getenv("COMPCLASS_TRAINING_VARIANT", "robust-v2-tta")

WORK_DIR = Path(os.getenv("COMPCLASS_WORK_DIR", Path.cwd())).expanduser().resolve()
DATA_DIR = WORK_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
SPLITS_DIR = DATA_DIR / "splits"
OUTPUTS_DIR = WORK_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
WEIGHTS_DIR = OUTPUTS_DIR / "weights"
METRICS_DIR = OUTPUTS_DIR / "metrics"

def _jsonable(value):
    if isinstance(value, dict):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    return value

def _fingerprint(samples):
    digest = hashlib.sha256()
    for sample in sorted(samples, key=lambda item: (item.label, item.path)):
        path = Path(sample.path)
        digest.update(f"{sample.label}|{path.name}|{path.stat().st_size}\n".encode())
    return digest.hexdigest()[:16]

def _split_fingerprint(splits):
    text = "\n".join(f"{name}|{sample.label}|{sample.path}" for name, rows in splits.items() for sample in rows)
    return hashlib.sha256(text.encode()).hexdigest()[:16]

def build_or_load_splits(samples, seed=SEED, ratios=SPLIT_RATIOS, force=False):
    if abs(sum(ratios) - 1.0) > 1e-6 or any(r <= 0 for r in ratios):
        raise ValueError(f"Invalid split ratios: {ratios}")
    for path in (RAW_DIR, SPLITS_DIR, FIGURES_DIR, WEIGHTS_DIR, METRICS_DIR):
        path.mkdir(parents=True, exist_ok=True)
    manifest = SPLITS_DIR / "split_manifest.json"
    fingerprint = _fingerprint(samples)
    if manifest.exists() and not force:
        try:
            payload = json.loads(manifest.read_text())
            matches = (
                payload.get("version") == 2
                and payload.get("dataset_fingerprint") == fingerprint
                and payload.get("seed") == seed
                and tuple(payload.get("ratios", [])) == tuple(ratios)
            )
            if matches:
                loaded = {
                    name: [
                        Sample(
                            str((WORK_DIR / Path(row["path"])).resolve()) if not Path(row["path"]).is_absolute() else row["path"],
                            row["label"],
                        )
                        for row in rows
                    ]
                    for name, rows in payload["splits"].items()
                }
                if all(Path(sample.path).is_file() for rows in loaded.values() for sample in rows):
                    console.print(f"Loaded frozen split from {manifest}")
                    return loaded
        except (OSError, KeyError, TypeError, json.JSONDecodeError):
            pass
        console.print("[yellow]Stale split detected; rebuilding it.[/yellow]")
    paths = [sample.path for sample in samples]
    labels = [sample.label for sample in samples]
    train_ratio, val_ratio, test_ratio = ratios
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        paths, labels, test_size=1.0 - train_ratio, random_state=seed, stratify=labels
    )
    relative_val = val_ratio / (val_ratio + test_ratio)
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=1.0 - relative_val, random_state=seed, stratify=temp_labels
    )
    splits = {
        "train": [Sample(path, label) for path, label in zip(train_paths, train_labels, strict=True)],
        "val": [Sample(path, label) for path, label in zip(val_paths, val_labels, strict=True)],
        "test": [Sample(path, label) for path, label in zip(test_paths, test_labels, strict=True)],
    }
    def serialise(sample):
        path = Path(sample.path).resolve()
        try:
            path = path.relative_to(WORK_DIR)
        except ValueError:
            pass
        return {"path": str(path), "label": sample.label}
    manifest.write_text(json.dumps({
        "version": 2,
        "dataset_fingerprint": fingerprint,
        "seed": seed,
        "ratios": list(ratios),
        "splits": {name: [serialise(sample) for sample in rows] for name, rows in splits.items()},
    }, indent=2))
    (SPLITS_DIR / "split_meta.json").write_text(json.dumps({
        "version": 2,
        "seed": seed,
        "ratios": list(ratios),
        "class_counts": {name: dict(Counter(sample.label for sample in rows)) for name, rows in splits.items()},
    }, indent=2))
    console.print(f"Wrote frozen split to {manifest}")
    return splits

def _limit(rows, maximum, split_name):
    if maximum is None:
        return list(rows)
    groups = defaultdict(list)
    for row in rows:
        groups[row.label].append(row)
    selected = []
    for label, group in sorted(groups.items()):
        group = sorted(group, key=lambda item: item.path)
        random.Random(f"{SEED}:{split_name}:{label}").shuffle(group)
        selected.extend(group[:maximum])
    return sorted(selected, key=lambda item: (item.label, item.path))

def apply_profile(splits):
    return {name: _limit(rows, PROFILE["max_per_class"], name) for name, rows in splits.items()}

def load_rgb(path, size=None):
    size = IMAGE_SIZE if size is None else size
    image = _open_rgb_image(path)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return resize(array, (size, size), anti_aliasing=True, preserve_range=True).astype(np.float32)

def hog_vector(image_rgb):
    return hog(rgb2gray(image_rgb), orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), block_norm="L2-Hys", feature_vector=True).astype(np.float32)

def extract_hog_matrix(samples, *, size=None, show_progress=True, stage="HOG"):
    labels = class_names if "class_names" in globals() else sorted({sample.label for sample in samples})
    label_to_idx = {label: index for index, label in enumerate(labels)}
    features, targets = [], []
    iterator = samples if not show_progress else tqdm(samples, desc=stage)
    for sample in iterator:
        features.append(hog_vector(load_rgb(sample.path, size)))
        targets.append(label_to_idx[sample.label])
    return np.stack(features), np.asarray(targets, dtype=np.int64), labels

def build_hog_svm(C=1.0):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svm", LinearSVC(C=C, max_iter=2000, dual=False, random_state=SEED)),
    ])

def build_transforms(train, size=None):
    size = IMAGE_SIZE if size is None else size
    steps = [transforms.Resize((size, size))]
    if train:
        if TRAINING_VARIANT in {"strong-regularized", "accuracy-regularized"}:
            steps.extend([transforms.RandomHorizontalFlip(), transforms.RandomRotation(20), transforms.ColorJitter(0.2, 0.2, 0.15), transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05))])
        else:
            steps.extend([transforms.RandomHorizontalFlip(), transforms.RandomRotation(15), transforms.ColorJitter(0.15, 0.15, 0.1)])
    steps.extend([transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    if train and TRAINING_VARIANT in {"strong-regularized", "accuracy-regularized"}:
        steps.append(transforms.RandomErasing(p=0.15, scale=(0.02, 0.08), ratio=(0.5, 2.0), value="random"))
    return transforms.Compose(steps)

def make_loaders(splits, labels, *, batch_size=None, num_workers=0):
    batch_size = BATCH_SIZE if batch_size is None else batch_size
    class_to_idx = {label: index for index, label in enumerate(labels)}
    loaders = {}
    for name, rows in splits.items():
        dataset = ComponentDataset(rows, class_to_idx, transform=build_transforms(name == "train"))
        loaders[name] = DataLoader(dataset, batch_size=batch_size, shuffle=name == "train", num_workers=num_workers, pin_memory=torch.cuda.is_available())
    return loaders

class SizedCNN(nn.Module):
    def __init__(self, num_classes, width=32, dropout=0.25):
        super().__init__()
        channels = [width, width * 2, width * 4, width * 6]
        blocks, in_channels = [], 3
        for out_channels in channels:
            blocks.extend([nn.Conv2d(in_channels, out_channels, 3, padding=1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True), nn.MaxPool2d(2)])
            in_channels = out_channels
        self.features = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(channels[-1], num_classes))
    def forward(self, inputs):
        return self.classifier(self.pool(self.features(inputs)))

def _replace_classifier(model, name, num_classes):
    if name.startswith("resnet"):
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name.startswith("densenet"):
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif name.startswith("mobilenet") or name.startswith("efficientnet") or name.startswith("convnext"):
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    elif name.startswith("swin"):
        model.head = nn.Linear(model.head.in_features, num_classes)
    else:
        raise ValueError(f"Unsupported backbone: {name}")
    return model


def build_backbone(name, num_classes):
    builders = {
        "resnet18": (models.resnet18, models.ResNet18_Weights.DEFAULT),
        "resnet34": (models.resnet34, models.ResNet34_Weights.DEFAULT),
        "resnet50": (models.resnet50, models.ResNet50_Weights.DEFAULT),
        "resnet101": (models.resnet101, models.ResNet101_Weights.DEFAULT),
        "resnet152": (models.resnet152, models.ResNet152_Weights.DEFAULT),
        "densenet121": (models.densenet121, models.DenseNet121_Weights.DEFAULT),
        "mobilenet_v3_large": (models.mobilenet_v3_large, models.MobileNet_V3_Large_Weights.DEFAULT),
        "mobilenet_v3_small": (models.mobilenet_v3_small, models.MobileNet_V3_Small_Weights.DEFAULT),
        "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.DEFAULT),
        "efficientnet_b2": (models.efficientnet_b2, models.EfficientNet_B2_Weights.DEFAULT),
        "efficientnet_v2_s": (models.efficientnet_v2_s, models.EfficientNet_V2_S_Weights.DEFAULT),
        "efficientnet_v2_m": (models.efficientnet_v2_m, models.EfficientNet_V2_M_Weights.DEFAULT),
        "convnext_tiny": (models.convnext_tiny, models.ConvNeXt_Tiny_Weights.DEFAULT),
        "convnext_small": (models.convnext_small, models.ConvNeXt_Small_Weights.DEFAULT),
        "convnext_base": (models.convnext_base, models.ConvNeXt_Base_Weights.DEFAULT),
        "swin_t": (models.swin_t, models.Swin_T_Weights.DEFAULT),
        "swin_v2_t": (models.swin_v2_t, models.Swin_V2_T_Weights.DEFAULT),
        "swin_b": (models.swin_b, models.Swin_B_Weights.DEFAULT),
    }
    if name not in builders:
        raise ValueError(f"Unsupported backbone: {name}")
    builder, default_weights = builders[name]
    try:
        model = builder(weights=default_weights if USE_PRETRAINED else None)
    except Exception as exc:
        if not ALLOW_PRETRAINED_FALLBACK:
            raise
        console.print(f"[yellow]Could not load {name} weights in smoke mode; using random initialisation: {exc}[/yellow]")
        model = builder(weights=None)
    return _replace_classifier(model, name, num_classes)
def _set_head_or_backbone(model, trainable):
    for name, parameter in model.named_parameters():
        head = name == "fc" or name.startswith("fc.") or name == "classifier" or name.startswith("classifier.") or name == "head" or name.startswith("head.")
        parameter.requires_grad = trainable or head

def train_torch_model(model, loaders, *, device, num_classes, epochs=10, lr=1e-3, weight_decay=1e-4, patience=3, freeze_backbone_epochs=0):
    model = model.to(device)
    train_targets = [loaders["train"].dataset.class_to_idx[sample.label] for sample in loaders["train"].dataset.samples]
    counts = torch.bincount(torch.tensor(train_targets), minlength=num_classes).float()
    class_weights = (counts.mean() / counts).sqrt().to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.08)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize(device)
        gpu_start, gpu_end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        gpu_start.record()
    else:
        gpu_start = gpu_end = None
    wall_start = time.perf_counter()
    if freeze_backbone_epochs:
        _set_head_or_backbone(model, False)
    optimizer = torch.optim.AdamW(filter(lambda parameter: parameter.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    best_f1, best_state, history, stale = -1.0, None, [], 0
    for epoch in range(1, epochs + 1):
        if freeze_backbone_epochs and epoch == freeze_backbone_epochs + 1:
            _set_head_or_backbone(model, True)
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr * 0.1, weight_decay=weight_decay)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
        model.train()
        loss_total, batches = 0.0, 0
        for inputs, targets in tqdm(loaders["train"], desc=f"epoch {epoch}/{epochs}", leave=False):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()
            loss_total += float(loss.item())
            batches += 1
        y_true, y_pred = predict_loader(model, loaders["val"], device)
        val_f1 = float(f1_score(y_true, y_pred, average="macro", labels=list(range(num_classes)), zero_division=0))
        history.append({"epoch": epoch, "train_loss": loss_total / max(batches, 1), "val_macro_f1": val_f1})
        scheduler.step(val_f1)
        console.print(f"epoch={epoch:02d} loss={history[-1]['train_loss']:.4f} val_macro_f1={val_f1:.4f}")
        if val_f1 > best_f1 + 1e-4:
            best_f1 = val_f1
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    if gpu_start is not None:
        gpu_end.record()
        torch.cuda.synchronize(device)
        gpu_seconds = gpu_start.elapsed_time(gpu_end) / 1000.0
        peak_memory = torch.cuda.max_memory_allocated(device) / 1e6
    else:
        gpu_seconds = peak_memory = None
    return {
        "model": model,
        "history": history,
        "best_val_macro_f1": best_f1,
        "wall_time_seconds": time.perf_counter() - wall_start,
        "gpu_time_seconds": gpu_seconds,
        "peak_gpu_memory_mb": peak_memory,
    }

def _experiment_fingerprint(name, settings):
    payload = {"name": name, "settings": settings, "profile": RUN_PROFILE, "seed": SEED, "image_size": IMAGE_SIZE, "classes": class_names, "split": SPLIT_SIGNATURE, "training_variant": TRAINING_VARIANT}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]

def run_torch(name, family, model, checkpoint, settings, loaders, **kwargs):
    fingerprint = _experiment_fingerprint(name, settings)
    if checkpoint.exists() and not FORCE_RETRAIN:
        payload = torch.load(checkpoint, map_location="cpu")
        if payload.get("metadata", {}).get("fingerprint") == fingerprint:
            model.load_state_dict(payload["state_dict"])
            console.print(f"Loaded {checkpoint.name}")
            return {"model": model.to(DEVICE), **payload.get("training", {}), "loaded": True}
    result = train_torch_model(model, loaders, **kwargs)
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "state_dict": {key: value.detach().cpu() for key, value in result["model"].state_dict().items()},
        "metadata": {
            "fingerprint": fingerprint, "model": name, "family": family, "settings": settings,
            "class_names": class_names, "image_size": IMAGE_SIZE, "mean": IMAGENET_MEAN,
            "std": IMAGENET_STD, "split_fingerprint": SPLIT_SIGNATURE,
        },
        "training": _jsonable({key: value for key, value in result.items() if key != "model"}),
    }, checkpoint)
    result["loaded"] = False
    return result

def run_hog(name, family, path, C, train_rows, val_rows):
    settings = {"C": C}
    fingerprint = _experiment_fingerprint(name, settings)
    if path.exists() and not FORCE_RETRAIN:
        payload = joblib.load(path)
        if payload.get("metadata", {}).get("fingerprint") == fingerprint:
            console.print(f"Loaded {path.name}")
            return payload["model"], payload.get("training", {})
    model = build_hog_svm(C)
    x_train, y_train, _ = extract_hog_matrix(train_rows, stage="HOG train")
    start = time.perf_counter()
    model.fit(x_train, y_train)
    x_val, y_val, _ = extract_hog_matrix(val_rows, stage="HOG validation")
    training = {
        "best_val_macro_f1": float(f1_score(y_val, model.predict(x_val), average="macro", labels=list(range(len(class_names))), zero_division=0)),
        "wall_time_seconds": time.perf_counter() - start,
        "gpu_time_seconds": None,
        "peak_gpu_memory_mb": None,
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump({
        "model": model,
        "metadata": {
            "fingerprint": fingerprint, "model": name, "family": family, "settings": settings,
            "class_names": class_names, "image_size": IMAGE_SIZE, "split_fingerprint": SPLIT_SIGNATURE,
        },
        "training": training,
    }, path)
    return model, training

def show_results(rows, title):
    frame = pd.DataFrame(rows)
    wanted = [column for column in ["model", "family", "accuracy", "macro_f1", "weighted_f1", "params", "wall_time_seconds", "gpu_time_seconds", "peak_gpu_memory_mb"] if column in frame.columns]
    frame = frame[wanted].sort_values("macro_f1", ascending=False).reset_index(drop=True)
    table = Table(title=title)
    for column in wanted:
        table.add_column(column)
    for row in frame.to_dict("records"):
        values = []
        for column in wanted:
            value = row[column]
            values.append("—" if value is None else f"{value:.4f}" if isinstance(value, float) else str(value))
        table.add_row(*values)
    console.print(table)
    display(frame)
    return frame

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(_jsonable(value), indent=2))


SESSION_MANIFEST_PATH = METRICS_DIR / "session_manifest.json"

def write_session_manifest(dataset_root, all_samples, all_splits, *, selected_splits=None, notebook_name=None):
    selected_splits = all_splits if selected_splits is None else selected_splits
    payload = {
        "contract_version": 1,
        "notebook": notebook_name,
        "profile": RUN_PROFILE,
        "seed": SEED,
        "split_ratios": list(SPLIT_RATIOS),
        "dataset_root": str(dataset_root),
        "dataset_fingerprint": _fingerprint(all_samples),
        "raw_split_manifest": str(SPLITS_DIR / "split_manifest_raw.json"),
        "raw_split_fingerprint": _split_fingerprint(all_splits),
        "raw_split_sizes": {name: len(rows) for name, rows in all_splits.items()},
        "profile_split_sizes": {name: len(rows) for name, rows in selected_splits.items()},
        "selection_metric": "raw_val_macro_f1",
        "test_policy": "Evaluate held-out test only after validation selection; never tune on test labels.",
        "force_retrain": FORCE_RETRAIN,
        "training_variant": TRAINING_VARIANT,
    }
    write_json(SESSION_MANIFEST_PATH, payload)
    console.print(f"session manifest={SESSION_MANIFEST_PATH}")
    return payload

ensure_output_dirs()
set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
console.print(Panel.fit(f"profile={RUN_PROFILE} · device={DEVICE} · image={IMAGE_SIZE}px · batch={BATCH_SIZE}\nwork={WORK_DIR}\ndata={DATA_DIR}\noutputs={OUTPUTS_DIR}", title="Component classification"))


## 2–3. Load data and run the fixed table


In [ ]:
dataset_root = resolve_dataset_root(prefer_kaggle=True)
all_samples = list_samples(dataset_root)
class_names = class_names_from_samples(all_samples)
all_splits = build_or_load_splits(all_samples, seed=SEED, ratios=SPLIT_RATIOS)
splits = apply_profile(all_splits)
SPLIT_SIGNATURE = _split_fingerprint(all_splits)
write_session_manifest(dataset_root, all_samples, all_splits, selected_splits=splits, notebook_name="02_playground")
loaders = make_loaders(splits, class_names, batch_size=BATCH_SIZE, num_workers=0)
summarize_splits(splits)

candidates = {}

from sklearn.ensemble import RandomForestClassifier
rf_settings = {"n_estimators": 10 if RUN_PROFILE == "smoke" else 300}
rf_fingerprint = _experiment_fingerprint("playground_hog_random_forest", rf_settings)
rf_checkpoint = WEIGHTS_DIR / "playground_hog_random_forest.joblib"
rf_start = time.perf_counter()
if rf_checkpoint.exists() and not FORCE_RETRAIN:
    rf_payload = joblib.load(rf_checkpoint)
    if rf_payload.get("metadata", {}).get("fingerprint") == rf_fingerprint:
        rf = rf_payload["model"]
        rf_elapsed = rf_payload.get("training", {}).get("wall_time_seconds")
    else:
        rf = None
else:
    rf = None
if rf is None:
    x_train, y_train, _ = extract_hog_matrix(splits["train"], stage="RF HOG train")
    x_val, y_val, _ = extract_hog_matrix(splits["val"], stage="RF HOG validation")
    rf = RandomForestClassifier(**rf_settings, n_jobs=-1, random_state=SEED)
    rf.fit(x_train, y_train)
    rf_elapsed = time.perf_counter() - rf_start
    joblib.dump({
        "model": rf,
        "metadata": {"fingerprint": rf_fingerprint, "model": "playground_hog_random_forest", "settings": rf_settings, "class_names": class_names, "split_fingerprint": SPLIT_SIGNATURE},
        "training": {"wall_time_seconds": rf_elapsed},
    }, rf_checkpoint)
x_val, y_val, _ = extract_hog_matrix(splits["val"], stage="RF HOG validation")
rf_values = compute_classification_metrics(y_val, rf.predict(x_val), class_names)
candidates["hog_random_forest"] = {"kind": "hog", "model": rf, "values": rf_values, "params": int(rf.n_estimators), "wall_time_seconds": rf_elapsed}


def record_torch(name, family, model, settings, **train_kwargs):
    result = run_torch(
        name, family, model, WEIGHTS_DIR / f"playground_{name}.pt", settings, loaders,
        device=DEVICE, num_classes=len(class_names), **train_kwargs,
    )
    true, predicted = predict_loader(result["model"], loaders["val"], DEVICE)
    values = compute_classification_metrics(true, predicted, class_names)
    candidates[name] = {
        "kind": "torch", "model": result["model"], "result": result,
        "values": values, "params": count_parameters(result["model"]),
    }


def run_sized_cnn(name, width):
    record_torch(
        name, "cnn_from_scratch", SizedCNN(len(class_names), width=width),
        {"width": width, "epochs": PROFILE["epochs"], "selection": "raw_val_macro_f1"},
        epochs=PROFILE["epochs"], patience=PROFILE["patience"], lr=1e-3,
    )


def run_backbone(name):
    record_torch(
        name, "transfer", build_backbone(name, len(class_names)),
        {"backbone": name, "pretrained": USE_PRETRAINED, "epochs": PROFILE["epochs"], "selection": "raw_val_macro_f1"},
        epochs=PROFILE["epochs"], patience=PROFILE["patience"], lr=1e-3, freeze_backbone_epochs=1,
    )


run_sized_cnn("tiny_cnn", 16)
run_sized_cnn("base_cnn", 32)
default_models = ("resnet18", "resnet34", "resnet50", "densenet121", "efficientnet_b0", "efficientnet_b2", "mobilenet_v3_large", "convnext_tiny")
MODEL_NAMES = tuple(filter(None, os.getenv("COMPCLASS_PLAYGROUND_MODELS", ",".join(default_models)).split(",")))
for name in MODEL_NAMES:
    run_backbone(name.strip())

rows = []
metrics_by_model = {}
for name, candidate in candidates.items():
    values = candidate["values"]
    row = {
        "model": name,
        "family": "classical" if candidate["kind"] == "hog" else "transfer" if name not in {"tiny_cnn", "base_cnn"} else "cnn_from_scratch",
        **values,
        "params": candidate["params"],
        "wall_time_seconds": candidate.get("wall_time_seconds") or candidate.get("result", {}).get("wall_time_seconds"),
        "gpu_time_seconds": candidate.get("result", {}).get("gpu_time_seconds"),
        "peak_gpu_memory_mb": candidate.get("result", {}).get("peak_gpu_memory_mb"),
    }
    rows.append(row)
    metrics_by_model[name] = values

selected_row = max(rows, key=lambda row: row["macro_f1"])
selected_name = selected_row["model"]
selected = candidates[selected_name]
if selected["kind"] == "hog":
    x_test, y_test, _ = extract_hog_matrix(splits["test"], stage="Selected HOG test")
    true_test, pred_test = y_test.tolist(), selected["model"].predict(x_test).tolist()
else:
    true_test, pred_test = predict_loader(selected["model"], make_loaders({"test": splits["test"]}, class_names, batch_size=BATCH_SIZE)["test"], DEVICE)
selected_test = compute_classification_metrics(true_test, pred_test, class_names)
plot_confusion_matrix(true_test, pred_test, class_names, title=f"Playground selected: {selected_name} — test", save_path=FIGURES_DIR / "cm_playground_selected.png")
show_results(rows, "Playground validation table")
console.print(f"selected by validation macro-F1: {selected_name} · test accuracy={selected_test['accuracy']:.4f}")
write_json(METRICS_DIR / "playground_metrics.json", {
    "run_profile": RUN_PROFILE, "seed": SEED, "split_ratios": SPLIT_RATIOS, "image_size": IMAGE_SIZE,
    "device": str(DEVICE), "dataset_fingerprint": _fingerprint(all_samples), "split_fingerprint": SPLIT_SIGNATURE,
    "class_names": class_names, "selection_metric": "raw_val_macro_f1", "test_policy": "test is evaluated only for the validation-selected row",
    "session_manifest": str(SESSION_MANIFEST_PATH), "validation_summary": rows, "metrics": metrics_by_model,
    "selected_model": selected_name, "selected_test_after_validation_selection": selected_test,
})


Future sandbox ideas: add one controlled row at a time for augmentation, resolution, frozen versus fully fine-tuned backbones, edge models, HOG settings, calibration, or a new dataset.
